<a href="https://colab.research.google.com/github/jnikem/pytorch-study/blob/main/sprint1_tensor_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import numpy as np

x = torch.tensor([1., 2., 3.])
print(x)
print(x.shape, x.dtype)

tensor([1., 2., 3.])
torch.Size([3]) torch.float32


In [4]:
a = np.arange(6).reshape(2, 3)

t1 = torch.from_numpy(a)                  # 메모리 공유 (a를 바꾸면 t1도 바뀜)
t2 = torch.tensor(a)                      # 복사
t3 = torch.tensor(a, dtype=torch.float32) # 실무에서 쓰는 형태

print(t1.dtype, t3.dtype)

torch.int64 torch.float32


In [5]:
z = torch.zeros(2, 3)
r = torch.randn(2, 3)          # np.random.randn과 동일
print(r[:, 0])                 # 열 뽑기 — 넘파이와 똑같다
print(r.sum(dim=0))            # axis → dim 으로 이름만 바뀜
print(r @ torch.randn(3, 4))   # 행렬곱도 그대로 @
print((r * 2).shape)           # 브로드캐스팅도 그대로

tensor([-1.1303,  0.5740])
tensor([-0.5563, -2.6821,  1.4027])
tensor([[-0.2609,  1.1556,  1.2576,  2.5953],
        [ 4.8433, -3.8726,  1.4235,  0.3302]])
torch.Size([2, 3])


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
r = r.to(device)

cpu


In [7]:
x = torch.tensor(3.0, requires_grad=True)

y = x**2 + 2*x + 1     # 여기서 녹화가 진행됨
print(y)               # grad_fn=<AddBackward0> ← 녹화 흔적

y.backward()           # 거꾸로 재생 = 미분
print(x.grad)          # dy/dx = 2x + 2, x=3 → 8.0

tensor(16., grad_fn=<AddBackward0>)
tensor(8.)


In [9]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

z = x * y + y**2
z.backward()

print(x.grad)   # dz/dx = y      = 3.0
print(y.grad)   # dz/dy = x + 2y = 8.0

tensor(3.)
tensor(8.)


In [10]:
w = torch.tensor(1.0, requires_grad=True)

for i in range(3):
  loss = w**2
  loss.backward()
  print(i, w.grad)

0 tensor(2.)
1 tensor(4.)
2 tensor(6.)


In [11]:
w = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    if w.grad is not None:
        w.grad.zero_()      # ← 이 한 줄이 없으면 학습이 조용히 망한다
    loss = w**2
    loss.backward()
    print(i, w.grad)        # 2.0, 2.0, 2.0

0 tensor(2.)
1 tensor(2.)
2 tensor(2.)


In [12]:
with torch.no_grad():
    y = x * 2      # grad_fn 없음, 메모리·속도 이득

In [15]:
# 과제 1. f(x) = 3x³ − 5x, x = 2에서 손으로 먼저 미분값을 구한 뒤 코드로 확인. 마크다운에 손계산 과정 한 줄 적기.
x = torch.tensor(2.0, requires_grad=True)

y = 3 * x ** 3 - 5 * x
y.backward()
print(x.grad)


tensor(31.)


In [17]:
# 과제 2. L = (w*x − y)² 에서 x=2, y=5, w=1일 때 dL/dw를 구하기. (힌트: 이게 다음 강의에서 할 선형회귀의 손실함수다)
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)
w = torch.tensor(1.0, requires_grad=True)

L = (w * x - y) ** 2
L.backward()
print(w.grad)

tensor(-12.)


In [18]:
# 과제 3. zero_grad()를 일부러 빼고 5번 반복 돌려서 grad가 어떻게 망가지는지 출력하고, 마크다운으로 왜 그런지 자기 말로 한 줄 설명.
x = torch.tensor(5.0, requires_grad=True)

for i in range(5):
  y = x ** 2 + 2 * x + 1
  y.backward()
  print(x.grad)

# 이유 : grad가 연산 과정을 녹화하고, 이를 backward 해서 미분값(변화율)을 구하는 과정은 누적식이다.
# 예를 들면 x^2+2x+1에서, x를 제곱하고, 2를 곱하고, 1을 더해준다 라는 변화를 backward에서 누적식으로 따라간다.
# grad를 초기화 해주지 않으면 이 누적이 계속 다음 연산에서도 이어져서 값이 쌓이는 게 아닐까 생각한다.

tensor(12.)
tensor(24.)
tensor(36.)
tensor(48.)
tensor(60.)
